# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imalik-7/Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Data contract — five plain-word answers

**1. What does one row mean?**  
In the raw warehouse table, one row represents one content item for one client on one day. For my lane, I will aggregate these daily rows so that my feature frame has one row per pseudonymized content item within a client.

**2. Which table will I use?**  
I will use `fact_content_daily_performance`. For this assignment I do not need the query-level table because I want to first build a small, explainable set of search-performance features.

**3. What time window will I use?**  
I will work only with the March 2026 partition. My feature window is March 1–15, 2026. My outcome window is March 16–30, 2026. March 31 is not used in this small experiment so both windows have equal length.

**4. What will I predict or rank?**  
I will use a temporary decline proxy: whether impressions in the 15-day outcome window are more than 20% lower than impressions in the preceding 15-day feature window. The eventual product output is a ranked list of pages that may deserve human review first.

**5. What will I deliberately exclude?**  
I will exclude all outcome-window measurements from the model features. In particular, `impressions_outcome15` cannot be used as a feature because it is measured after the decision moment and is used to construct the label.

In [15]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

In [16]:
import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata
from huggingface_hub import whoami

# Read your token from Colab Secrets.
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Open the Secrets/key panel in Colab "
        "and add a secret named HF_TOKEN."
    )

# Check that the token actually belongs to a valid Hugging Face account.
account = whoami(token=HF_TOKEN)

print("Hugging Face authentication successful.")
print("Username:", account["name"])

Hugging Face authentication successful.
Username: imalik7


In [17]:
# Create DuckDB connection.
con = duckdb.connect()

# Safely create Hugging Face secret inside DuckDB.
safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{safe_token}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

# IMPORTANT:
# We use March 2026, not the final June sample.
MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connection prepared.")
print("Working month: March 2026")

DuckDB connection prepared.
Working month: March 2026


In [18]:
test_rows = con.sql(f"""
SELECT *
FROM {MARCH}
LIMIT 5
""").df()

print("Warehouse connection works.")
display(test_rows)

Warehouse connection works.


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [19]:
schema = con.sql(f"""
DESCRIBE SELECT *
FROM {MARCH}
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [20]:
WAREHOUSE_COLUMNS = schema["column_name"].tolist()

print("Number of columns:", len(WAREHOUSE_COLUMNS))
print(WAREHOUSE_COLUMNS)

Number of columns: 31
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [5]:
print("Source grain: report_date × client_hash_id × content_hash_id")
print("Feature-frame grain: one client-content item")
print("Feature window: 2026-03-01 to 2026-03-15")
print("Outcome window: 2026-03-16 to 2026-03-30")
print("Lane: Refresh / Content Opportunity Scoring")

Source grain: report_date × client_hash_id × content_hash_id
Feature-frame grain: one client-content item
Feature window: 2026-03-01 to 2026-03-15
Outcome window: 2026-03-16 to 2026-03-30
Lane: Refresh / Content Opportunity Scoring


### Field classification

| Field | Bucket | Why |
|---|---|---|
| `impressions_feature15` | Feature | Knowable at the decision moment because it uses only March 1–15 search impressions. |
| `clicks_feature15` | Feature | Knowable at the decision moment because it uses only March 1–15 clicks. |
| `ctr_feature15` | Feature | Knowable at the decision moment because it is calculated only from March 1–15 clicks and impressions. |
| `avg_position_feature15` | Feature | Knowable at the decision moment because it uses only March 1–15 search-position measurements. |
| `active_impression_days_feature15` | Feature | Knowable at the decision moment because it counts only March 1–15 days with impressions. |
| `is_declining_proxy` | Label / proxy | This is the temporary outcome I am trying to predict/rank. |
| `client_hash_id` | Context | Used for grouping and validation, not as a model feature. |
| `content_hash_id` | Context | Identifies the content item but has no predictive meaning itself. |
| `impressions_outcome15` | Excluded | It comes from the future/outcome window and directly helps construct the label. |

In [6]:
FEATURES = [
    "impressions_feature15",
    "clicks_feature15",
    "ctr_feature15",
    "avg_position_feature15",
    "active_impression_days_feature15",
]

LABEL = "is_declining_proxy"

CONTEXT = [
    "client_hash_id",
    "content_hash_id",
]

EXCLUDED = [
    "impressions_outcome15",
]

print("Feature count:", len(FEATURES))
print("Features:", FEATURES)
print("Label:", LABEL)
print("Context:", CONTEXT)
print("Excluded:", EXCLUDED)

assert len(FEATURES) == 5

Feature count: 5
Features: ['impressions_feature15', 'clicks_feature15', 'ctr_feature15', 'avg_position_feature15', 'active_impression_days_feature15']
Label: is_declining_proxy
Context: ['client_hash_id', 'content_hash_id']
Excluded: ['impressions_outcome15']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
# VERIFICATION QUERY 1 — grain check

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_at_grain
    FROM {MARCH}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

print("Duplicate grain rows found:", len(grain_check))
display(grain_check)

if len(grain_check) == 0:
    print("PASS: report_date × client × content is unique in this slice.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,rows_at_grain


PASS: report_date × client × content is unique in this slice.


In [22]:
# VERIFICATION QUERY 2 — row count and date window

slice_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items
    FROM {MARCH}
""").df()

display(slice_summary)

,row_count,min_date,max_date,clients,content_items
0,9841378,2026-03-01,2026-03-31,55,331437


In [23]:
# VERIFICATION QUERY 3 — analytics availability

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,

        ROUND(
            100.0 *
            COUNT(*) FILTER (
                WHERE ga4_data_available IS TRUE
            )
            / COUNT(*),
            2
        ) AS ga4_available_pct

    FROM {MARCH}
""").df()

display(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_available_pct
0,9841378,413966,4.21


In [24]:
# Build one row per client-content item.
# Features use March 1-15.
# Outcome uses March 16-30.

feature_frame = con.sql(f"""
    WITH aggregated AS (

        SELECT
            client_hash_id,
            content_hash_id,

            -- FEATURE 1
            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS impressions_feature15,

            -- FEATURE 2
            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS clicks_feature15,

            -- FEATURE 4: impression-weighted average position
            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                    THEN gsc_avg_position * gsc_impressions
                    ELSE 0
                END
            )
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN report_date BETWEEN DATE '2026-03-01'
                                             AND DATE '2026-03-15'
                        THEN gsc_impressions
                        ELSE 0
                    END
                ),
                0
            ) AS avg_position_feature15,

            -- FEATURE 5
            COUNT(
                DISTINCT CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                         AND gsc_impressions > 0
                    THEN report_date
                END
            ) AS active_impression_days_feature15,

            -- OUTCOME WINDOW — NEVER A MODEL FEATURE
            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-16'
                                         AND DATE '2026-03-30'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS impressions_outcome15

        FROM {MARCH}

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        *,

        -- FEATURE 3
        100.0 * clicks_feature15
        / NULLIF(impressions_feature15, 0)
        AS ctr_feature15,

        -- TEMPORARY LABEL / PROXY
        CASE
            WHEN impressions_outcome15
                 < 0.80 * impressions_feature15
            THEN 1
            ELSE 0
        END AS is_declining_proxy

    FROM aggregated

    -- Enough initial visibility to reduce tiny-volume noise
    WHERE impressions_feature15 >= 100

""").df()

print("Feature-frame rows:", len(feature_frame))
print(
    "Unique client-content pairs:",
    feature_frame[
        ["client_hash_id", "content_hash_id"]
    ].drop_duplicates().shape[0]
)

display(
    feature_frame[
        CONTEXT + FEATURES + [LABEL]
    ].head(10)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 77540
Unique client-content pairs: 77540


,client_hash_id,content_hash_id,impressions_feature15,clicks_feature15,ctr_feature15,avg_position_feature15,active_impression_days_feature15,is_declining_proxy
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,0.143781,6.265037,15,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,0.000000,4.085714,15,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,0.080972,6.297706,15,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,0.327869,7.370902,15,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,0.416667,3.770833,15,1
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,131.0,0.0,0.000000,9.809160,15,1
6,client_73cda7b4e4f265ea,content_22c063002b7c1caf,172.0,0.0,0.000000,7.755814,15,1
7,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,3104.0,16.0,0.515464,5.542848,15,0
8,client_73cda7b4e4f265ea,content_20403327d8d9374c,1294.0,5.0,0.386399,6.990726,15,0
9,client_73cda7b4e4f265ea,content_f8df6b20d18c4374,527.0,1.0,0.189753,7.387097,15,1


### Why these five features are available at decision time

- **impressions_feature15:** known because all impressions were observed during March 1–15, before the outcome window.
- **clicks_feature15:** known because all clicks were observed during March 1–15.
- **ctr_feature15:** known because it is calculated only from feature-window clicks and impressions.
- **avg_position_feature15:** known because it uses only search-position observations from the feature window.
- **active_impression_days_feature15:** known because it counts only feature-window days where the page had impressions.

The decision moment is after March 15. None of these five features use measurements from March 16–30.

In [25]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import balanced_accuracy_score

model_data = feature_frame.dropna(
    subset=["is_declining_proxy"]
).copy()

X = model_data[FEATURES]
y = model_data[LABEL]
groups = model_data["client_hash_id"]

# Client-grouped split: clients do not cross train/test.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(
        max_depth=4,
        random_state=42
    )
)

honest_model.fit(
    X.iloc[train_idx],
    y.iloc[train_idx]
)

honest_pred = honest_model.predict(
    X.iloc[test_idx]
)

honest_score = balanced_accuracy_score(
    y.iloc[test_idx],
    honest_pred
)

print(
    f"HONEST balanced accuracy: "
    f"{honest_score:.3f}"
)

HONEST balanced accuracy: 0.499


In [26]:
# DELIBERATE LEAKAGE EXPERIMENT
#
# This column uses the OUTCOME WINDOW.
# It must never be a real feature.

model_data["leaky_outcome_ratio"] = (
    model_data["impressions_outcome15"]
    / model_data["impressions_feature15"]
)

LEAKY_FEATURES = FEATURES + [
    "leaky_outcome_ratio"
]

X_leaky = model_data[LEAKY_FEATURES]

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(
        max_depth=4,
        random_state=42
    )
)

leaky_model.fit(
    X_leaky.iloc[train_idx],
    y.iloc[train_idx]
)

leaky_pred = leaky_model.predict(
    X_leaky.iloc[test_idx]
)

leaky_score = balanced_accuracy_score(
    y.iloc[test_idx],
    leaky_pred
)

print(
    f"HONEST score: {honest_score:.3f}"
)

print(
    f"LEAKED score: {leaky_score:.3f}"
)

print(
    f"Artificial improvement: "
    f"{leaky_score - honest_score:+.3f}"
)

HONEST score: 0.499
LEAKED score: 1.000
Artificial improvement: +0.501


In [27]:
# Remove the deliberately leaked column.
model_data.drop(
    columns=["leaky_outcome_ratio"],
    inplace=True
)

# Final honest frame: context + exactly five features + label.
final_frame = model_data[
    CONTEXT + FEATURES + [LABEL]
].copy()

print("Leak removed.")
print("Final feature count:", len(FEATURES))
print("Final features:", FEATURES)
print(f"Honest score kept: {honest_score:.3f}")

display(final_frame.head())

Leak removed.
Final feature count: 5
Final features: ['impressions_feature15', 'clicks_feature15', 'ctr_feature15', 'avg_position_feature15', 'active_impression_days_feature15']
Honest score kept: 0.499


,client_hash_id,content_hash_id,impressions_feature15,clicks_feature15,ctr_feature15,avg_position_feature15,active_impression_days_feature15,is_declining_proxy
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,0.143781,6.265037,15,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,0.000000,4.085714,15,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,0.080972,6.297706,15,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,0.327869,7.370902,15,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,0.416667,3.770833,15,1


### Leakage lesson

The first model used only five measurements that were available before the decision moment. I then deliberately created `leaky_outcome_ratio`, which used impressions from the outcome window that also defines the decline proxy.

After adding that column, the quick validation score moved toward a nearly perfect result. That improvement is not real model skill — the feature contains information from the answer.

I therefore deleted the leaked column and kept the honest score from the five pre-decision features. This experiment shows why a high model score is not trustworthy until the feature and target windows have been checked for leakage.

## 4. Data limits

## Data limitation

**Single-month limitation:** this experiment uses only March 2026 and two short 15-day windows. It can show a useful workflow for defining features and a decline proxy, but it cannot establish long-term decline, seasonality, or whether refreshing a page would cause performance to recover.

The result should therefore be treated as directional decision-support for which pages may deserve review, not as proof that an intervention will improve a page.

In [28]:
print("Named limitation: single-month March 2026 slice")
print("This notebook does not claim causation or long-term seasonality.")

Named limitation: single-month March 2026 slice
This notebook does not claim causation or long-term seasonality.


## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.